In [28]:
import polars as pl
import json
import random
from collections import defaultdict
from langdetect import detect
from itertools import groupby
import nltk
import spacy
import json
import benepar
import time

In [29]:
dataset = pl.read_parquet("data_test.parquet")


In [30]:
# for data in dataset:
text = dataset["content.text"]
para = dataset["content.annotations.paragraph"]
fields = list(set(dataset["field"]))
content = []
for i, field in enumerate(fields):
    text = dataset.filter(pl.col('field') == field).select(pl.col('content.text'))[:10].to_dicts()
    para = dataset.filter(pl.col('field') == field).select(pl.col('content.annotations.paragraph'))[:10].to_dicts()

    paragraph = ""

    for i,row in enumerate(para):
        rand = len(row)
        num = random.randrange(rand)
        ch = json.loads(row["content.annotations.paragraph"])[num]

        data = text[i]["content.text"][int(ch["start"]):int(ch["end"])].strip() + " "
        if detect(data) == "en":
            paragraph += data
    
    content.append({field : paragraph})

with open('input_content.json', 'w') as f:
    json.dump(content, f)

In [31]:
def nltk_split():
    start_time = time.time()
    result = []

    for paper in content:
        for key,values in paper.items():
            sent_text = nltk.sent_tokenize(values)
            result.append({key : sent_text})
            
    end_time =  time.time()
    
    return start_time - end_time

# with open('nltk_result.json', 'w') as f:
#     json.dump(result, f)

In [32]:
def spacy_split():
    start_time = time.time()
    nlp = spacy.load('en_core_web_lg')
    nlp.add_pipe('sentencizer')

    result = []

    for paper in content:
        for key, values in paper.items():
            doc = nlp(values)
            sents = []
            for sentence in doc.sents:
                sents.append(sentence.text) 
                result.append({key: sents})
            
    end_time =  time.time()
    
    return start_time - end_time
            
    # with open('spacy_result.json', 'w') as f:
    #     json.dump(result, f)

In [33]:
def spacy_benepar():
    start_time = time.time()
    nlp = spacy.load('en_core_web_lg')
    nlp.add_pipe('benepar', config={'model': 'benepar_en3'})
    result = []

    for paper in content:
        for key, values in paper.items():
            doc = nlp(values)
            sents = []
            for sentence in doc.sents:
                doc = nlp(sentence.text)
                if (list(doc.sents)[0]._.labels[0] == "S"):
                    sents.append(sentence.text) 
            result.append({key: sents})
    
    end_time =  time.time()
    return start_time - end_time
        
# with open('beneparSpacy_result.json', 'w') as f:
#     json.dump(result, f)

In [34]:
def scispacy_split():
    start_time = time.time()

    nlp = spacy.load("en_core_sci_scibert")

    result = []

    for paper in content:
        for key, values in paper.items():
            doc = nlp(values)
            sents = []
            for sentence in doc.sents:
                sents.append(sentence.text) 
                result.append({key: sents})
    
    end_time =  time.time()
    return start_time - end_time
        
# with open('scispacy_result.json', 'w') as f:
#     json.dump(result, f)

In [35]:

def scispacy_benepar():
    start_time = time.time()

    nlp = spacy.load("en_core_sci_scibert")

    nlp.add_pipe('benepar', config={'model': 'benepar_en3'})

    result = []

    for paper in content:
        for key, values in paper.items():
            doc = nlp(values)
            sents = []
            for sentence in doc.sents:
                doc = nlp(sentence.text)
                if (list(doc.sents)[0]._.labels[0] == "S"):
                    sents.append(sentence.text) 
            result.append({key: sents})
            
    end_time =  time.time()
    return start_time - end_time
        
# with open('beneparScispacy_result.json', 'w') as f:
#     json.dump(result, f)

In [36]:
def scispacy_plus():
    start_time = time.time()

    nlp = spacy.load("en_core_sci_scibert")

    result = []
    allCount = 0
    for paper in content:
        for key, values in paper.items():
            doc = nlp(values)
            sents = []
            count = 0
            for sentence in doc.sents:
                init = sentence.text.strip()[0]
                sentCond = (init.isalpha() and init.isupper() or init == '(' or init == '[' or init.isdigit())
                if (len(sentence.text.split(" ")) >= 3 and sentence.text.strip()[-1] in [".", "!", "?"] and sentCond):
                    sents.append(sentence.text)

            result.append({key: sents})
            
    end_time =  time.time()
    return start_time - end_time
        
# with open('scispacyPlus_result.json', 'w') as f:
#     json.dump(result, f)

In [37]:
def avg(lst):
    return sum(lst) / len(lst)

In [38]:
n_list = []
spacy_list = []
spacy_b = []
sciscpacy_list=[]
scispacy_b=[]
scispacy_p =[]
for i in range(5):
    n_list.append(nltk_split())
    spacy_list.append(spacy_split())
    sciscpacy_list.append(scispacy_split())
    spacy_b.append(spacy_benepar())
    scispacy_b.append(scispacy_benepar())
    scispacy_p.append(scispacy_plus())
    
print("-------- sum --------")
print("nltk::: ", sum(n_list))
print("spacy::: ", sum(spacy_list))
print("spacy benepar::: ", sum(spacy_b))
print("sciscpacy::: ", sum(sciscpacy_list))
print("scispacy benepar::: ", sum(scispacy_b))
print("scispacy plus::: ", sum(scispacy_p))  

print("-------- avg --------")
print("nltk::: ", avg(n_list))
print("spacy::: ", avg(spacy_list))
print("spacy benepar::: ", avg(spacy_b))
print("sciscpacy::: ", avg(sciscpacy_list))
print("scispacy benepar::: ", avg(scispacy_b))
print("scispacy plus::: ", avg(scispacy_p))  

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
/home/pop533611/anaconda3/envs/ph/lib/python3.9/site-packages/torch/distributions/distribution.py:44: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(f'{self.__class__} does not define `arg_constraints`. ' +
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed

-------- sum --------
nltk:::  -0.13776397705078125
spacy:::  -19.458430290222168
spacy benepar:::  -261.8229560852051
sciscpacy:::  -67.92988300323486
scispacy benepar:::  -404.79104471206665
scispacy plus:::  -65.6815094947815
-------- avg --------
nltk:::  -0.02755279541015625
spacy:::  -3.8916860580444337
spacy benepar:::  -52.36459121704102
sciscpacy:::  -13.585976600646973
scispacy benepar:::  -80.95820894241334
scispacy plus:::  -13.1363018989563
